# Unsupervised learning for patient phenotypes

Exercise notebook for the lesson **[Unsupervised learning for patient phenotypes](https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Unsupervised learning looks for structure in data without an outcome label, for example grouping patients into phenotypes. Clustering of six clinical variables has suggested subgroups of adult-onset diabetes in Swedish and Indian populations. But clustering algorithms always return clusters, even from pure noise, so any proposed phenotype must be shown to be stable, reproducible in other populations and more useful than simple clinical measures.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Clustering synthetic patients

Three synthetic diabetes profiles with overlapping distributions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(17)
centres = {
    "deficient": dict(age_dx=45, bmi=25, hba1c=10.5, homa_b=40, homa_ir=2.0),
    "resistant": dict(age_dx=50, bmi=34, hba1c=7.5, homa_b=120, homa_ir=5.5),
    "age_related": dict(age_dx=66, bmi=27, hba1c=7.0, homa_b=85, homa_ir=2.5),
}
spread = dict(age_dx=8, bmi=3, hba1c=1.0, homa_b=20, homa_ir=0.9)
frames = [pd.DataFrame({k: rng.normal(v, spread[k], 600) for k, v in c.items()})
          .assign(true_group=name) for name, c in centres.items()]
df = pd.concat(frames, ignore_index=True)
variables = list(spread)
print(df.groupby("true_group")[variables].mean().round(1))

#              age_dx   bmi  hba1c  homa_b  homa_ir
# true_group
# age_related    65.9  26.9    7.0    84.1      2.5
# deficient      44.6  25.2   10.5    38.9      2.0
# resistant      50.4  34.1    7.6   121.2      5.5

### Choosing the number of clusters

Continuing from the previous block: silhouette scores for k = 2 to 6.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df[variables])
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f"k = {k}: silhouette {silhouette_score(X, labels):.2f}")

# k = 2: silhouette 0.39
# k = 3: silhouette 0.45
# k = 4: silhouette 0.34
# k = 5: silhouette 0.25
# k = 6: silhouette 0.16

### Checking against the truth

Continuing from the previous block: compare k = 3 clusters with the true groups.

In [ ]:
from sklearn.metrics import adjusted_rand_score

labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
print(pd.crosstab(df.true_group, labels, colnames=["cluster"]))
print(f"\nAgreement with true groups (adjusted Rand index) "
      f"{adjusted_rand_score(df.true_group, labels):.2f}")

# cluster        0    1    2
# true_group
# age_related    4  590    6
# deficient      2    1  597
# resistant    596    4    0
#
# Agreement with true groups (adjusted Rand index) 0.97

### Clustering always finds clusters

k-means on data with no structure, and a stability check.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

rng = np.random.default_rng(3)
noise = rng.normal(size=(1800, 5))                 # no groups at all
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(noise)
print(f"Cluster sizes on pure noise: {np.bincount(labels)}")
print(f"Silhouette {silhouette_score(noise, labels):.2f}")

# Stability: refit on two random halves and compare assignments on the full data
a, b = (KMeans(n_clusters=4, n_init=10, random_state=s)
        .fit(noise[rng.choice(1800, 900, replace=False)]) for s in (1, 2))
print(f"Agreement between two refits (adjusted Rand index) "
      f"{adjusted_rand_score(a.predict(noise), b.predict(noise)):.2f}")

# Cluster sizes on pure noise: [454 446 445 455]
# Silhouette 0.13
# Agreement between two refits (adjusted Rand index) 0.20

## Exercise

**Goal:** Cluster synthetic diabetes profiles, test whether the clusters are real and stable, and compare them with continuous variables for predicting an outcome.

**Data:** Synthetic profiles generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the silhouette scores and the comparison with the true groups.

In [ ]:
# Your code here


### Step 2

Make the groups overlap more (increase the spreads) and watch how silhouette and agreement change.

In [ ]:
# Your code here


### Step 3

Run a bootstrap stability analysis: refit k-means on 50 resamples and compute the average adjusted Rand index.

In [ ]:
# Your code here


### Step 4

Try Gaussian mixture models and hierarchical clustering; do they agree with k-means?

In [ ]:
# Your code here


### Step 5

Simulate a complication outcome that depends on HbA1c and HOMA-IR, and compare a model using cluster labels with one using the continuous variables.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping](https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.